# Whisper Bangla LoRA — DGX Spark Full Run

`--full` switches the profile from the Colab rehearsal to the real run:
`openai/whisper-large-v3`, **bf16**, batch 32, 10 epochs, **r=128 / alpha=256**.
Same scripts, same collator — only `configs/*.yaml`'s `full` profile applies.

Prerequisite: the Colab debug checklist passed end to end on whisper-small.


## 0 · Setup

In [ ]:
!nvidia-smi
# Prefer the compiled lock; fall back to direct pins if it hasn't been built yet.
!test -f requirements/train-linux-cu124.txt   && pip install -q -r requirements/train-linux-cu124.txt   || pip install -q -r requirements/train.in
import os
os.environ.setdefault("HF_HOME", "./data/raw/hf_cache")


## 1 · Build the full dataset
No `--limit`: every source is downloaded in full, then mixed 20/40/25/15.

In [ ]:
for src in ["medibeng", "indicvoices", "common_voice", "fleurs"]:
    !python scripts/02_prepare_data.py --source {src}
!python scripts/03_augment.py --source medibeng --n-augments 4
!python scripts/04_build_dataset.py


## 2 · Baseline WER before training
Gives the number the fine-tune is measured against (~35% Bengali / ~45% mixed expected).

In [ ]:
!python scripts/06_eval.py --full --limit 200

## 3 · Full LoRA training (~1.5-2 hrs)

First large-v3 run on this box — bf16, gradient checkpointing, early stopping
patience 3, MLflow tracking. Features are re-extracted at 128 mel bins (the T4
run used 80), so do not reuse a cached feature set from Colab.


In [ ]:
!python scripts/05_train.py --full

## 4 · Eval the tuned adapter
Targets: ~15-20% WER Bengali clean, ~20-28% on BD-English mixed.

In [ ]:
!python scripts/06_eval.py --full --adapter outputs/whisper-bangla-lora

## 5 · Export for production serving

In [ ]:
!python scripts/07_export.py --adapter outputs/whisper-bangla-lora